In [1]:
import pandas as pd
import tqdm, os
import pickle
import warnings
warnings.simplefilter("ignore")

import sys
sys.path.insert(0, "../..")

from utils.acoustics_processing import get_mfccs
from utils.articulation_processing import get_aemri75_articulatory_features

In [2]:
df = pd.read_csv("../data_processing/dataset_construction/datasets/processed_aemri75.csv")
df["subject"] = df["art_f"].apply(lambda x: os.path.basename(x).split("_")[0])

In [3]:
df

,phone1_start,phone1_end,phone2_start,phone2_end,phone1,phone2,context,word,word_id,art_f,ac_f,subject
0,1.85,1.87,1.87,2.04,ð,ɛ,ð,then,0,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,sub043
1,1.87,2.04,2.04,2.07,ɛ,n,ð_ɛ,then,0,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,sub043
2,2.07,2.09,2.09,2.14,ð,ʌ,ð,the,1,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,sub043
3,2.14,2.18,2.18,2.24,n,ɔ,n,north,2,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,sub043
4,2.18,2.24,2.24,2.39,ɔ,ɹ,n_ɔ,north,2,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,sub043
...,...,...,...,...,...,...,...,...,...,...,...,...
26851,52.92,52.97,52.97,53.01,æ,ŋ,l_æ,language,134,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,sub071
26852,52.97,53.01,53.01,53.04,ŋ,g,l_æ_ŋ,language,134,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,sub071
26853,53.01,53.04,53.04,53.10,g,w,l_æ_ŋ_g,language,134,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,sub071
26854,53.04,53.10,53.10,53.28,w,ɨ,l_æ_ŋ_g_w,language,134,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,sub071


In [ ]:
tokens = {
    "start": [],
    "end": [],
    "syllable": [],
    "art_f": [],
    "ac_f": [],
    "subject": [],
}

metadata = set(
    [(start, end, target, art_f, ac_f, subject) for start, end, target, art_f, ac_f, subject in zip(df["syllable1_start"], df["syllable1_end"], df["syllable1"], df["art_f"], df["ac_f"], df["subject"])] + 
    [(start, end, target, art_f, ac_f, subject) for start, end, target, art_f, ac_f, subject in zip(df["syllable2_start"], df["syllable2_end"], df["syllable2"], df["art_f"], df["ac_f"], df["subject"])]
)

metadata = sorted(list(metadata), key=lambda x: (x[3], x[4], x[5]))

for start, end, target, art_f, ac_f, subject in metadata:
    tokens["start"].append(start)
    tokens["end"].append(end)
    tokens["syllable"].append(target)
    tokens["art_f"].append(art_f)
    tokens["ac_f"].append(ac_f)
    tokens["subject"].append(subject)

In [5]:
token_df = pd.DataFrame(tokens)

In [6]:
token_df

,start,end,phone,art_f,ac_f,subject
0,1.85,1.87,ð,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,sub043
1,1.87,2.04,ɛ,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,sub043
2,2.07,2.09,ð,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,sub043
3,2.14,2.18,n,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,sub043
4,2.18,2.24,ɔ,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,sub043
...,...,...,...,...,...,...
37844,51.81,52.07,i,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,sub071
37845,52.48,52.51,n,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,sub071
37846,52.70,52.75,n,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,sub071
37847,52.80,52.88,z,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,sub071


In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from utils.models import VideoAutoencoder, SequenceAutoencoder

In [8]:
def latent_stats(z):
    # z: (B, latent_dim)
    return {
        "latent_var_mean": z.var(dim=0).mean().item(),
        "latent_var_min":  z.var(dim=0).min().item(),
        "latent_var_max":  z.var(dim=0).max().item(),
    }

def batch_diversity(x):
    # x: (B, ...)
    x_flat = x.reshape(x.size(0), -1)
    return x_flat.std(dim=0).mean().item()

def visualize(clean, recon, name="Reconstruction"):
    import matplotlib.pyplot as plt

    try:
        # Get both channels, frame 0 of first batch element
        clean_mri= clean[0, 0, 0].cpu().numpy()
    except RuntimeError:
        clean_mri = clean[0, 0, 0].detach().cpu().numpy()

    recon_mri = recon[0, 0, 0].detach().cpu().numpy()

    plt.figure(figsize=(10, 6))

    plt.subplot(1, 2, 1)
    plt.imshow(clean_mri, cmap='gray')
    plt.title("Original (MRI)")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(recon_mri, cmap='gray')
    plt.title(f"{name} (MRI)")
    plt.axis('off')

    plt.tight_layout()
    plt.savefig(f"video_visualization/aemri75/{name.lower()}.png", dpi=300)
    plt.close("all")

def combined_loss(y_true, y_pred, warped_x, criterion, plot=False, ratio=1):
    diff_true = y_true[:, :, 1:, :, :] - y_true[:, :, :-1, :, :]
    diff_pred = y_pred[:, :, 1:, :, :] - y_pred[:, :, :-1, :, :]
    loss = criterion(diff_true, diff_pred) * (1 - ratio) + criterion(y_true, y_pred) * ratio

    if plot:
        visualize(diff_true, diff_pred, name="Reconstruction (differential)")
        visualize(y_true, y_pred, name="Reconstruction")
        visualize(y_true, warped_x, name="Transformation")

    return loss

In [9]:
device = "mps" if torch.backends.mps.is_available() else "cpu"

BATCH_SIZE = 4

In [ ]:
log = {
    "batch_idx": [],
    "loss": [],
}

art_model = VideoAutoencoder(
    input_shape=(5, 84, 84),
    latent_dim=1024,
    use_mlp=True,
    grid_sample_cpu_fallback=True,
).to(device)

art_criterion = nn.MSELoss()
art_optimizer = torch.optim.Adam(art_model.parameters(), lr=0.001)

shuffled_token_df = token_df.sample(frac=1, random_state=42).reset_index(drop=True)

pbar = tqdm.tqdm(shuffled_token_df.iterrows(), total=len(shuffled_token_df))

art_model.train()

batch = []
batch_idx = 0
for idx, (_, data) in enumerate(pbar):

    start, end, phone, art_f, ac_f, subject = data["start"], data["end"], data["phone"], data["art_f"], data["ac_f"], data["subject"]
    
    articulatory_features = get_aemri75_articulatory_features(art_f, start, end)

    if articulatory_features is None:
        continue

    batch.append(
        torch.tensor(articulatory_features, dtype=torch.float32)
    )

    # --------------------------------------------------
    # When batch is ready
    # --------------------------------------------------
    if len(batch) < BATCH_SIZE:
        continue
    batch_idx += 1

    art_batch = torch.stack(batch).to(device)  # (B,T,P,C)
    batch.clear()

    # ===== Articulatory =====
    art_out, art_warped_x, art_z = art_model(art_batch, return_warped_x=True, return_latent=True)
    art_loss = combined_loss(art_batch, art_out, art_warped_x, art_criterion, plot=batch_idx % 30 == 0, ratio=0.5)

    art_optimizer.zero_grad()
    art_loss.backward()
    art_optimizer.step()

    log["batch_idx"].append(batch_idx)
    log["loss"].append(art_loss.item())
    pbar.set_description(
        f"{batch_idx}: ART {art_loss.item():.4f}"
    )

    # --------------------------------------------------
    # Diagnostics every 1000 updates
    # --------------------------------------------------
    if (batch_idx) % 300 == 0:
        with torch.no_grad():
            art_stats = latent_stats(art_z)

        print(
            f"[ART] latent var mean={art_stats['latent_var_mean']:.4e}, "
            f"min={art_stats['latent_var_min']:.4e}"
        )

        print(
            f"[ART] input std={batch_diversity(art_batch):.4e}, "
            f"recon std={batch_diversity(art_out):.4e}"
        )

        torch.save(art_model.state_dict(), "ckpts/aemri75/art_model_ckpt.pth")
        pd.DataFrame(log).to_csv("ckpts/aemri75/art_model_training_log.csv", index=False)

torch.save(art_model.state_dict(), "ckpts/aemri75/art_model_ckpt.pth")
pd.DataFrame(log).to_csv("ckpts/aemri75/art_model_training_log.csv", index=False)

300: ART 0.3435:   3%|▎         | 1201/37849 [01:14<34:21, 17.78it/s] 

[ART] latent var mean=3.2465e+04, min=8.5481e-01
[ART] input std=3.8901e-01, recon std=8.9561e-02


600: ART 0.3062:   6%|▋         | 2406/37849 [02:27<32:00, 18.45it/s]  

[ART] latent var mean=1.4431e+05, min=3.7508e-01
[ART] input std=4.2950e-01, recon std=5.6029e-02


900: ART 0.3073:  10%|▉         | 3614/37849 [03:41<1:05:58,  8.65it/s]

...

9000: ART 0.2937:  96%|█████████▌| 36173/37849 [36:49<03:04,  9.07it/s]

[ART] latent var mean=1.0216e+07, min=2.3808e+02
[ART] input std=4.2393e-01, recon std=8.3373e-02


9300: ART 0.3884:  99%|█████████▉| 37376/37849 [38:01<01:12,  6.49it/s]

[ART] latent var mean=3.8489e+06, min=3.8365e+01
[ART] input std=3.6559e-01, recon std=1.1204e-01


9417: ART 0.3045: 100%|██████████| 37849/37849 [38:28<00:00, 16.39it/s]


In [13]:
def temporal_diff(x):
    """
    x: (B, T, ...)
    returns: (B, T-1, ...)
    """
    return x[:, 1:] - x[:, :-1]

def combined_loss(x_hat, x, alpha=0.5):
    """
    alpha: weight on static loss (delta weight = 1-alpha)
    """
    # static loss
    loss_static = F.mse_loss(x_hat, x)

    # delta loss
    dx_hat = temporal_diff(x_hat)
    dx = temporal_diff(x)

    loss_delta = F.mse_loss(dx_hat, dx)

    return alpha * loss_static + (1 - alpha) * loss_delta, loss_static, loss_delta

In [14]:
BATCH_SIZE = 4

In [ ]:
log = {
    "batch_idx": [],
    "loss": [],
}
ac_model = SequenceAutoencoder(
    feature_shape=30,
    latent_dim=1024,
).to(device)

ac_criterion = nn.MSELoss()
ac_optimizer = torch.optim.Adam(ac_model.parameters(), lr=0.001)

shuffled_token_df = token_df.sample(frac=1, random_state=42).reset_index(drop=True)

pbar = tqdm.tqdm(shuffled_token_df.iterrows(), total=len(shuffled_token_df))

ac_model.train()

batch  = []
batch_idx = 0
for idx, (_, data) in enumerate(pbar):

    start, end, phone, art_f, ac_f, subject = data["start"], data["end"], data["phone"], data["art_f"], data["ac_f"], data["subject"]
    try:
        mfccs = get_mfccs(ac_f, start, end).T
    except:
        continue

    if mfccs is None:
        continue

    batch.append(
        torch.tensor(mfccs, dtype=torch.float32)
    )

    # --------------------------------------------------
    # When batch is ready
    # --------------------------------------------------
    if len(batch) < BATCH_SIZE:
        continue
    batch_idx += 1
    ac_batch = torch.stack(batch).to(device)  # (B,T,F)
    batch.clear()

    # ===== Acoustic =====
    ac_out, ac_z = ac_model(ac_batch)
    ac_loss, ac_static, ac_delta = combined_loss(
        ac_out, ac_batch
    )

    ac_optimizer.zero_grad()
    ac_loss.backward()
    ac_optimizer.step()

    log["batch_idx"].append(batch_idx)
    log["loss"].append(ac_loss.item())
    pbar.set_description(
        f"{batch_idx}: AC {ac_loss.item():.4f}"
    )

    # --------------------------------------------------
    # Diagnostics every 1000 updates
    # --------------------------------------------------
    if batch_idx % 1000 == 999:
        with torch.no_grad():
            ac_stats  = latent_stats(ac_z)

        print(
            f"[AC ] latent var mean={ac_stats['latent_var_mean']:.4e}, "
            f"min={ac_stats['latent_var_min']:.4e}"
        )
        print(
            f"[AC ] input std={batch_diversity(ac_batch):.4e}, "
            f"recon std={batch_diversity(ac_out):.4e}"
        )

        torch.save(ac_model.state_dict(), "ckpts/aemri75/ac_model_ckpt.pth")
        pd.DataFrame(log).to_csv("ckpts/aemri75/ac_model_training_log.csv", index=False)

torch.save(ac_model.state_dict(), "ckpts/aemri75/ac_model_ckpt.pth")
pd.DataFrame(log).to_csv("ckpts/aemri75/ac_model_training_log.csv", index=False)

2999: AC 145.6374:   8%|▊         | 2984/37849 [00:13<02:39, 218.31it/s]

[AC ] latent var mean=3.5476e+01, min=1.4575e-04
[AC ] input std=1.3906e+01, recon std=4.3140e+00


3999: AC 118.5625:  11%|█         | 3996/37849 [00:18<02:27, 229.16it/s]

[AC ] latent var mean=3.4706e+00, min=5.1584e-06
[AC ] input std=1.1828e+01, recon std=1.5815e+00


5011: AC 114.2406:  13%|█▎        | 4996/37849 [00:23<02:23, 229.37it/s]

...

32015: AC 118.6359:  85%|████████▍ | 32012/37849 [02:54<00:48, 120.36it/s]

[AC ] latent var mean=4.5717e+03, min=3.9599e-03
[AC ] input std=1.1628e+01, recon std=5.0055e+00


37015: AC 124.5420:  98%|█████████▊| 37004/37849 [03:26<00:07, 118.59it/s]

[AC ] latent var mean=1.4335e+04, min=2.7175e-03
[AC ] input std=1.3030e+01, recon std=7.1650e+00


37845: AC 112.6115: 100%|██████████| 37849/37849 [03:31<00:00, 178.93it/s]
